In [1]:
import gc
import os

import numpy as np
import pandas as pd
import torch
from lightning import pytorch as pl
from lightning.pytorch.callbacks import ModelCheckpoint
from lightning.pytorch.loggers import TensorBoardLogger
from torch.utils.data import DataLoader

from benchmarks import LightningModule, MultiTargetScaler, PrintEpochMetricsCallback
from benchmarks.cnn import CNN
from dataset.pac_matrix import PAC5MatrixDataset

torch.set_float32_matmul_precision("medium")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


k_folds = 5
target_properties = [
    "xtb_homo",
    "xtb_lumo",
    "xtb_hl_gap",
    "pm7_homo",
    "pm7_lumo",
    "pm7_hl_gap",
    "pm7_hof",
]
property_types = [
    "intensive",
    "intensive",
    "intensive",
    "intensive",
    "intensive",
    "intensive",
    "extensive",
]
dataset = "PAC5-3"
model_path_template = f"../models/cnn_1.1M-pac_grid-[{dataset}]-fold_{{fold}}"
model_logs_dir = "../logs"
model_dataset_dir = "../dataset"
model_logs_template = f"cnn_1.1M-pac_grid-[{dataset}]-fold_{{fold}}"
num_tasks = len(target_properties)
loss_weights = torch.ones(num_tasks, device=device)

W0930 10:45:25.031000 22392 .venv\Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


Using device: cuda


In [2]:
fold_results = []

for fold in range(1, k_folds + 1):
    print(f"\n======[ Fold {fold}/{k_folds} ]======")

    train_path = model_dataset_dir + f"/{dataset}/fold_{fold}_train.csv"
    val_path = model_dataset_dir + f"/{dataset}/fold_{fold}_val.csv"

    train_dataset = PAC5MatrixDataset(
        train_path, properties=target_properties, num_workers=20
    )
    val_dataset = PAC5MatrixDataset(
        val_path, properties=target_properties, num_workers=20
    )

    scaler = MultiTargetScaler()
    scaler.fit(train_dataset.targets.numpy())

    train_dataset.targets = torch.tensor(
        scaler.transform(train_dataset.targets.numpy()), dtype=torch.float32
    )
    val_dataset.targets = torch.tensor(
        scaler.transform(val_dataset.targets.numpy()), dtype=torch.float32
    )

    train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=256, shuffle=False)

    model = CNN(
        property_types=property_types, kernel_size=5, num_hidden=3, hidden_channels=92
    )
    lightning_model = LightningModule(model, target_properties, scaler, lr=1e-3)

    save_path_dir = model_path_template.format(fold=fold)
    os.makedirs(save_path_dir, exist_ok=True)

    checkpoint_callback = ModelCheckpoint(
        dirpath=save_path_dir,
        filename="best-{epoch}-{val_loss:.5f}",
        monitor="val_loss",
        mode="min",
        save_top_k=1,
    )

    logger = TensorBoardLogger(
        save_dir=model_logs_dir, name=model_logs_template.format(fold=fold)
    )

    trainer = pl.Trainer(
        max_epochs=100,
        gradient_clip_val=1.0,
        gradient_clip_algorithm="norm",
        accelerator="auto",
        logger=logger,
        devices=1,
        callbacks=[checkpoint_callback, PrintEpochMetricsCallback(target_properties)],
        enable_checkpointing=True,
        num_sanity_val_steps=0,
        log_every_n_steps=1,
        precision="bf16-mixed" if device.type == "cuda" else "32-true",
        enable_progress_bar=False,
    )

    trainer.fit(lightning_model, train_loader, val_loader)

    best_model_score = checkpoint_callback.best_model_score
    if best_model_score is None:
        raise RuntimeError("The checkpoint callback did not record a validation loss.")
    best_val_loss = best_model_score.item()
    print(f"Best scaled validation loss for fold {fold}: {best_val_loss:.4f}")
    fold_results.append(best_val_loss)

    best_model_path = checkpoint_callback.best_model_path
    best_lightning_model = LightningModule.load_from_checkpoint(
        best_model_path,
        model=model,
        property_names=target_properties,
        scaler=scaler,
        enable_progress_bar=False,
    )
    torch.save(best_lightning_model.model.state_dict(), save_path_dir + "/best.ckpt")

    lightning_model.cpu()
    best_lightning_model.cpu()

    del train_dataset, val_dataset, train_loader, val_loader
    del model, lightning_model, best_lightning_model
    del trainer, checkpoint_callback, logger

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("\n======[ Final k-fold results ]======")
print(f"Scaled Loss per fold: {[round(float(x), 4) for x in fold_results]}")
print(f"Average Scaled Loss: {np.mean(fold_results):.4f} ± {np.std(fold_results):.4f}")


======[ Fold 1/5 ]======


Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Loading `train_dataloader` to estimate number of stepping batches.
D:\Coding\pac-gcnn\.venv\Lib\site-packages\lightning\pytorch\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
D:\Coding\pac-gcnn\.venv\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=27` in the `DataLoader` to improve performance.
D:\Coding\pac-gcnn\.venv\L

┏━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name       ┃ Type       ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model      │ CNN        │  1.1 M │ train │     0 │
│ 1 │ criterion  │ MSELoss    │      0 │ train │     0 │
│ 2 │ train_rmsd │ ModuleDict │      0 │ train │     0 │
│ 3 │ val_rmsd   │ ModuleDict │      0 │ train │     0 │
│ 4 │ test_rmsd  │ ModuleDict │      0 │ train │     0 │
│ 5 │ test_mae   │ ModuleDict │      0 │ train │     0 │
│ 6 │ test_r2    │ ModuleDict │      0 │ train │     0 │
│ 7 │ test_max   │ ModuleDict │      0 │ train │     0 │
│ 8 │ test_min   │ ModuleDict │      0 │ train │     0 │
└───┴────────────┴────────────┴────────┴───────┴───────┘

Trainable params: 1.1 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 1.1 M                                                                                                
Total estimated model params size (MB): 4.530                                                                      
Modules in train mode: 80                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

D:\Coding\pac-gcnn\.venv\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=27` in the `DataLoader` to improve performance.


...

┏━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name       ┃ Type       ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model      │ CNN        │  1.1 M │ train │     0 │
│ 1 │ criterion  │ MSELoss    │      0 │ train │     0 │
│ 2 │ train_rmsd │ ModuleDict │      0 │ train │     0 │
│ 3 │ val_rmsd   │ ModuleDict │      0 │ train │     0 │
│ 4 │ test_rmsd  │ ModuleDict │      0 │ train │     0 │
│ 5 │ test_mae   │ ModuleDict │      0 │ train │     0 │
│ 6 │ test_r2    │ ModuleDict │      0 │ train │     0 │
│ 7 │ test_max   │ ModuleDict │      0 │ train │     0 │
│ 8 │ test_min   │ ModuleDict │      0 │ train │     0 │
└───┴────────────┴────────────┴────────┴───────┴───────┘

Trainable params: 1.1 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 1.1 M                                                                                                
Total estimated model params size (MB): 4.530                                                                      
Modules in train mode: 80                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0


Detected KeyboardInterrupt, attempting graceful shutdown ...


SystemExit: 1

D:\Coding\pac-gcnn\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3783: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [2]:
metric_keys = ["mae", "rmsd", "max", "min", "r2"]
all_results = {
    "train": {
        property_name: {metric: [] for metric in metric_keys}
        for property_name in target_properties
    },
    "test": {
        property_name: {metric: [] for metric in metric_keys}
        for property_name in target_properties
    },
}

for fold in range(1, k_folds + 1):
    print(f"\n======[ Evaluating fold {fold}/{k_folds} ]======")

    train_path = model_dataset_dir + f"/{dataset}/fold_{fold}_train.csv"
    test_path = model_dataset_dir + f"/{dataset}/fold_{fold}_test.csv"

    train_dataset = PAC5MatrixDataset(
        train_path, properties=target_properties, num_workers=6
    )
    test_dataset = PAC5MatrixDataset(
        test_path, properties=target_properties, num_workers=6
    )

    scaler = MultiTargetScaler()
    scaler.fit(train_dataset.targets.numpy())

    train_dataset.targets = torch.tensor(
        scaler.transform(train_dataset.targets.numpy()), dtype=torch.float32
    )
    test_dataset.targets = torch.tensor(
        scaler.transform(test_dataset.targets.numpy()), dtype=torch.float32
    )

    train_loader = DataLoader(train_dataset, batch_size=256, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False)

    model = CNN(
        property_types=property_types, kernel_size=5, num_hidden=3, hidden_channels=92
    )
    load_path = model_path_template.format(fold=fold) + "/best.ckpt"
    model.load_state_dict(torch.load(load_path, map_location=device))

    lightning_model = LightningModule(model, target_properties, scaler)

    trainer = pl.Trainer(
        accelerator="auto", devices=1, logger=False, enable_progress_bar=False
    )

    intercepted_preds = []

    def capture_predictions(module, input, output):
        intercepted_preds.append(output.detach().cpu().numpy())

    hook_handle = model.register_forward_hook(capture_predictions)

    print("\n======[ Train Set Evaluation ]======")
    intercepted_preds.clear()
    train_results = trainer.test(
        lightning_model, dataloaders=train_loader, verbose=False
    )[0]
    train_preds_scaled = np.vstack(intercepted_preds)

    print("\n======[ Test Set Evaluation ]======")
    intercepted_preds.clear()
    test_results = trainer.test(
        lightning_model, dataloaders=test_loader, verbose=False
    )[0]
    test_preds_scaled = np.vstack(intercepted_preds)

    hook_handle.remove()

    train_preds_unscaled = scaler.inverse_transform(train_preds_scaled)
    test_preds_unscaled = scaler.inverse_transform(test_preds_scaled)

    train_targets_unscaled = scaler.inverse_transform(train_dataset.targets.numpy())
    test_targets_unscaled = scaler.inverse_transform(test_dataset.targets.numpy())

    train_df = pd.DataFrame()
    test_df = pd.DataFrame()

    for i, prop in enumerate(target_properties):
        train_df[f"{prop}_true"] = train_targets_unscaled[:, i]
        train_df[f"{prop}_pred"] = train_preds_unscaled[:, i]

        test_df[f"{prop}_true"] = test_targets_unscaled[:, i]
        test_df[f"{prop}_pred"] = test_preds_unscaled[:, i]

    for i, prop in enumerate(target_properties):
        print(f"======[ {prop} ]======")
        t_mae = train_results[f"test_mae_{prop}"]
        t_rmsd = train_results[f"test_rmsd_{prop}"]
        t_max = train_results[f"test_max_{prop}"]
        t_min = train_results[f"test_min_{prop}"]
        t_r2 = train_results[f"test_r2_{prop}"]
        print(
            f" + Train -> MAE: {t_mae:.4f}\tRMSD: {t_rmsd:.4f}\tMax: {t_max:.4f}\tMin: {t_min:.4f}\tR^2: {t_r2:.4f}"
        )

        te_mae = test_results[f"test_mae_{prop}"]
        te_rmsd = test_results[f"test_rmsd_{prop}"]
        te_max = test_results[f"test_max_{prop}"]
        te_min = test_results[f"test_min_{prop}"]
        te_r2 = test_results[f"test_r2_{prop}"]
        print(
            f" + Test -> MAE: {te_mae:.4f}\tRMSD: {te_rmsd:.4f}\tMax: {te_max:.4f}\tMin: {te_min:.4f}\tR^2: {te_r2:.4f}"
        )

        for metric in metric_keys:
            all_results["train"][prop][metric].append(
                train_results[f"test_{metric}_{prop}"]
            )
            all_results["test"][prop][metric].append(
                test_results[f"test_{metric}_{prop}"]
            )

    del model
    del lightning_model
    del trainer
    del scaler
    del train_results
    del test_results
    del train_loader
    del test_loader
    del train_dataset
    del test_dataset
    del train_df
    del test_df
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("\n======[ Cross-validation results (mean ± std) ]======")

for property_name in target_properties:
    print(f"\n======[ {property_name} ]======")

    train_statistics = {
        metric: (
            np.mean(all_results["train"][property_name][metric]),
            np.std(all_results["train"][property_name][metric], ddof=1),
        )
        for metric in metric_keys
    }

    print(
        f" + Train -> "
        f"MAE: {train_statistics['mae'][0]:.4f} ± {train_statistics['mae'][1]:.4f}\t"
        f"RMSD: {train_statistics['rmsd'][0]:.4f} ± {train_statistics['rmsd'][1]:.4f}\t"
        f"Max: {train_statistics['max'][0]:.4f} ± {train_statistics['max'][1]:.4f}\t"
        f"Min: {train_statistics['min'][0]:.4f} ± {train_statistics['min'][1]:.4f}\t"
        f"R^2: {train_statistics['r2'][0]:.4f} ± {train_statistics['r2'][1]:.4f}"
    )

    test_statistics = {
        metric: (
            np.mean(all_results["test"][property_name][metric]),
            np.std(all_results["test"][property_name][metric], ddof=1),
        )
        for metric in metric_keys
    }

    print(
        f" + Test -> "
        f"MAE: {test_statistics['mae'][0]:.4f} ± {test_statistics['mae'][1]:.4f}\t"
        f"RMSD: {test_statistics['rmsd'][0]:.4f} ± {test_statistics['rmsd'][1]:.4f}\t"
        f"Max: {test_statistics['max'][0]:.4f} ± {test_statistics['max'][1]:.4f}\t"
        f"Min: {test_statistics['min'][0]:.4f} ± {test_statistics['min'][1]:.4f}\t"
        f"R^2: {test_statistics['r2'][0]:.4f} ± {test_statistics['r2'][1]:.4f}"
    )


======[ Evaluating fold 1/5 ]======


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
D:\Coding\pac-gcnn\.venv\Lib\site-packages\lightning\pytorch\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
D:\Coding\pac-gcnn\.venv\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:434: The 'test_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` 


======[ Train Set Evaluation ]======


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]



======[ Test Set Evaluation ]======
======[ xtb_homo ]======
 + Train -> MAE: 0.0181	RMSD: 0.0235	Max: 0.2382	Min: 0.0000	R^2: 0.9970
 + Test -> MAE: 0.0340	RMSD: 0.0530	Max: 0.9086	Min: 0.0000	R^2: 0.9850
======[ xtb_lumo ]======
 + Train -> MAE: 0.0199	RMSD: 0.0258	Max: 0.2951	Min: 0.0000	R^2: 0.9977
 + Test -> MAE: 0.0366	RMSD: 0.0562	Max: 0.7891	Min: 0.0000	R^2: 0.9888
======[ xtb_hl_gap ]======
 + Train -> MAE: 0.0203	RMSD: 0.0262	Max: 0.3112	Min: 0.0000	R^2: 0.9960
 + Test -> MAE: 0.0467	RMSD: 0.0745	Max: 1.4427	Min: 0.0000	R^2: 0.9673
======[ pm7_homo ]======
 + Train -> MAE: 0.0232	RMSD: 0.0304	Max: 0.4008	Min: 0.0000	R^2: 0.9964
 + Test -> MAE: 0.0450	RMSD: 0.0676	Max: 1.2347	Min: 0.0000	R^2: 0.9823
======[ pm7_lumo ]======
 + Train -> MAE: 0.0179	RMSD: 0.0232	Max: 0.2951	Min: 0.0000	R^2: 0.9956
 + Test -> MAE: 0.0366	RMSD: 0.0562	Max: 2.8291	Min: 0.0000	R^2: 0.9741
======[ pm7_hl_gap ]======
 + Train -> MAE: 0.0237	RMSD: 0.0310	Max: 0.3926	Min: 0.0000	R^2: 0.9953
 + Test -> 

...


======[ Test Set Evaluation ]======
======[ xtb_homo ]======
 + Train -> MAE: 0.0174	RMSD: 0.0225	Max: 0.1869	Min: 0.0000	R^2: 0.9973
 + Test -> MAE: 0.0346	RMSD: 0.0542	Max: 1.1984	Min: 0.0000	R^2: 0.9842
======[ xtb_lumo ]======
 + Train -> MAE: 0.0192	RMSD: 0.0248	Max: 0.2584	Min: 0.0000	R^2: 0.9978
 + Test -> MAE: 0.0370	RMSD: 0.0574	Max: 1.1534	Min: 0.0000	R^2: 0.9884
======[ xtb_hl_gap ]======
 + Train -> MAE: 0.0188	RMSD: 0.0242	Max: 0.3247	Min: 0.0000	R^2: 0.9966
 + Test -> MAE: 0.0471	RMSD: 0.0760	Max: 1.3019	Min: 0.0000	R^2: 0.9662
======[ pm7_homo ]======
 + Train -> MAE: 0.0203	RMSD: 0.0270	Max: 0.3872	Min: 0.0000	R^2: 0.9972
 + Test -> MAE: 0.0446	RMSD: 0.0678	Max: 1.5828	Min: 0.0000	R^2: 0.9822
======[ pm7_lumo ]======
 + Train -> MAE: 0.0165	RMSD: 0.0215	Max: 0.2367	Min: 0.0000	R^2: 0.9962
 + Test -> MAE: 0.0371	RMSD: 0.0567	Max: 1.1107	Min: 0.0000	R^2: 0.9738
======[ pm7_hl_gap ]======
 + Train -> MAE: 0.0204	RMSD: 0.0270	Max: 0.3758	Min: 0.0000	R^2: 0.9965
 + Test -> 

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
D:\Coding\pac-gcnn\.venv\Lib\site-packages\lightning\pytorch\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
D:\Coding\pac-gcnn\.venv\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:434: The 'test_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` 


======[ Train Set Evaluation ]======


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]



======[ Test Set Evaluation ]======
======[ xtb_homo ]======
 + Train -> MAE: 0.0180	RMSD: 0.0235	Max: 0.3752	Min: 0.0000	R^2: 0.9970
 + Test -> MAE: 0.0340	RMSD: 0.0532	Max: 1.1320	Min: 0.0000	R^2: 0.9849
======[ xtb_lumo ]======
 + Train -> MAE: 0.0202	RMSD: 0.0262	Max: 0.6085	Min: 0.0000	R^2: 0.9976
 + Test -> MAE: 0.0368	RMSD: 0.0573	Max: 1.2567	Min: 0.0000	R^2: 0.9885
======[ xtb_hl_gap ]======
 + Train -> MAE: 0.0201	RMSD: 0.0260	Max: 0.3632	Min: 0.0000	R^2: 0.9960
 + Test -> MAE: 0.0465	RMSD: 0.0753	Max: 1.6083	Min: 0.0000	R^2: 0.9666
======[ pm7_homo ]======
 + Train -> MAE: 0.0220	RMSD: 0.0292	Max: 0.3362	Min: 0.0000	R^2: 0.9967
 + Test -> MAE: 0.0447	RMSD: 0.0681	Max: 2.0767	Min: 0.0000	R^2: 0.9821
======[ pm7_lumo ]======
 + Train -> MAE: 0.0176	RMSD: 0.0230	Max: 0.3181	Min: 0.0000	R^2: 0.9957
 + Test -> MAE: 0.0369	RMSD: 0.0568	Max: 2.5794	Min: 0.0000	R^2: 0.9737
======[ pm7_hl_gap ]======
 + Train -> MAE: 0.0224	RMSD: 0.0297	Max: 0.3188	Min: 0.0000	R^2: 0.9957
 + Test -> 